# Prototype Diligence Copilot

This notebook is a notebook-first prototype for a commercial due diligence copilot focused on insurance and climate-risk markets.

## What This Notebook Does

It walks through one simple RAG-style workflow:

1. select a company folder
2. extract text from local PDFs
3. split the text into chunks
4. create embeddings and save them once
5. reload saved chunks
6. retrieve relevant chunks for a question
7. ask the LLM to answer using only retrieved evidence

## How To Use It

- Change `company_name` in the config cell.
- Run the notebook top to bottom the first time for a company.
- On later runs for the same company, skip the one-time embedding cell and load the saved JSON file instead.


## 1. Setup

Load packages, your API key, and the company-specific configuration used throughout the notebook.

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from pathlib import Path
import os
import json
import math

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

model_name = "gpt-5-mini"
embedding_model = "text-embedding-3-small"
client = OpenAI(api_key=api_key)

# Change this one value to run the notebook for a different company.
company_name = "Guidewire"
company_folder = Path(f"Data/{company_name}")
chunks_file = f"{company_name.lower()}_chunks.json"

True


## 2. Extract And Chunk Documents

Read all PDFs in the selected company folder and split them into overlapping text chunks.

Why chunking matters:
- full filings are too large to send every time
- retrieval works better on smaller units of text
- embeddings are usually created per chunk, not per full document


In [9]:
# Extract text from each PDF and split it into overlapping chunks.

pdf_files = list(company_folder.glob("*.pdf"))
print([pdf_file.name for pdf_file in pdf_files])

def extract_pdf_chunks(pdf_path, max_chars=3000, overlap=300):
    reader = PdfReader(pdf_path)
    chunks = []

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        i = 0

        while i < len(text):
            chunk_text = text[i:i + max_chars]

            chunks.append({
                "file_name": pdf_path.name,
                "page_number": page_num,
                "text": chunk_text
            })

            i += max_chars - overlap

    return chunks

all_chunks = []
for pdf_file in pdf_files:
    all_chunks.extend(extract_pdf_chunks(pdf_file))

print(len(all_chunks))


['10-Q.pdf', 'IR.pdf', '10-K.pdf']
514


## 3. One-Time Embedding Step

This is the expensive part. Run it once per company to create and save embeddings locally.

After that, you can reuse the saved JSON file and skip regenerating embeddings unless the source documents change.

In [ ]:
# Create embeddings for every chunk and save them locally.

for chunk in all_chunks:
    response = client.embeddings.create(
        model=embedding_model,
        input=chunk["text"]
    )
    chunk["embedding"] = response.data[0].embedding

with open(chunks_file, "w") as f:
    json.dump(all_chunks, f)

print(f"Saved {len(all_chunks)} chunks to {chunks_file}")


## 4. Reload Saved Chunks

Reload the saved chunk file so future notebook runs are much faster and cheaper.

In [10]:
# Load saved chunks so you do not need to regenerate embeddings every run.

with open(chunks_file, "r") as f:
    saved_chunks = json.load(f)

print(len(saved_chunks))
print(saved_chunks[0].keys())


514
dict_keys(['file_name', 'page_number', 'text', 'embedding'])


## 5. Retrieval

Use embedding similarity to find the most relevant chunks for a user question.

This is the retrieval part of RAG.

In [11]:
# Retrieve the most relevant chunks for a question using embedding similarity.

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (norm_a * norm_b)

def search_chunks_by_embedding(chunks, query, top_k=5):
    query_response = client.embeddings.create(
        model=embedding_model,
        input=query
    )
    query_embedding = query_response.data[0].embedding

    results = []
    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        results.append({
            "file_name": chunk["file_name"],
            "page_number": chunk["page_number"],
            "text": chunk["text"],
            "score": score
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


## 6. Inspect Retrieved Evidence

Before asking the model to answer, quickly inspect the top retrieved chunks for one sample question.

This is helpful for debugging retrieval quality and checking whether the pipeline is grounding the model in the right evidence.

In [ ]:
# Check the top retrieved chunks for one question.

query = f"What products does {company_name} sell to insurance companies?"
results = search_chunks_by_embedding(saved_chunks, query)

for result in results:
    print("score:", result["score"])
    print("file:", result["file_name"])
    print("page:", result["page_number"])
    print(result["text"][:500])
    print("-" * 80)


## 7. Answer Questions With Retrieved Context

The function below retrieves the most relevant chunks, builds a context block, and asks the LLM to answer using only that context.

In [12]:
# Ask one diligence question using the retrieved chunks as context.

def answer_question(saved_chunks, query, top_k=5):
    results = search_chunks_by_embedding(saved_chunks, query, top_k=top_k)

    context = "\n\n".join(
        [
            f"File: {r['file_name']}, Page: {r['page_number']}\n{r['text']}"
            for r in results
        ]
    )

    prompt = f"""
        Use only the context below to answer the question.
        If the context is not sufficient, say what is missing.
        Include file and page references when making factual claims.

        Question:
        {query}

        Context:
        {context}

        Please provide:
        - a short answer
        - 3 bullet summary points
        - 3 possible commercial risks
    """

    response = client.responses.create(
        model=model_name,
        input=prompt
    )

    return {
        "query": query,
        "answer": response.output_text,
        "results": results,
    }


## 8. Run A First-Pass Diligence Question Set

These standard questions act like a lightweight first-pass diligence brief for the selected company.

In [ ]:
# Standard first-pass diligence questions for the selected company.

queries = [
    f"What products does {company_name} sell to insurance companies?",
    f"Who are {company_name}'s target customers and buyer types?",
    f"What are the main commercial risks for {company_name}?",
    f"What evidence is there for {company_name}'s competitive strengths or differentiation?",
    f"What follow-up commercial diligence questions should an investor ask about {company_name}?",
]

for query in queries:
    qa_result = answer_question(saved_chunks, query, top_k=5)
    print(f"QUESTION: {qa_result['query']}")
    print()
    print(qa_result["answer"])
    print('\n' + '=' * 100 + '\n')


## Notes

- This is still a simple prototype.
- Retrieval is based on saved chunk embeddings plus cosine similarity.
- There is no vector database yet.
- There is no web ingestion yet.
- There are no agents yet.

That is intentional: the goal of this notebook is to keep the workflow easy to understand while proving the core idea.

In [13]:
brief_questions = {
    "Company overview": f"What does {company_name} do and what does it sell?",
    "Customers": f"Who are {company_name}'s target customers and buyer types?",
    "Commercial risks": f"What are the main commercial risks for {company_name}?",
    "Competitive strengths": f"What evidence is there for {company_name}'s competitive strengths or differentiation?",
    "Follow-up diligence questions": f"What follow-up commercial diligence questions should an investor ask about {company_name}?"
}

brief = {}

for section, query in brief_questions.items():
    result = answer_question(saved_chunks, query, top_k=5)
    brief[section] = result["answer"]

print(f"FIRST-PASS DILIGENCE BRIEF: {company_name}")
print("=" * 80)

for section, answer in brief.items():
    print()
    print(section.upper())
    print("-" * len(section))
    print(answer)

FIRST-PASS DILIGENCE BRIEF: Guidewire

COMPANY OVERVIEW
----------------
Short answer
- Guidewire provides a cloud-native core software platform and related digital, analytics and AI products for property & casualty (P&C) insurers, and sells those products primarily as cloud-based subscription services (e.g., InsuranceSuite and InsuranceNow) plus complementary data, marketplace apps and services (10-K.pdf, p.7; 10-K.pdf, p.11; 10-K.pdf, p.15).

3‑point summary
- Core offering: subscription cloud platforms for P&C insurers — InsuranceSuite (PolicyCenter, ClaimCenter, BillingCenter, etc.) and InsuranceNow — that act as insurers’ transactional systems of record for policy, claims and billing (10-K.pdf, p.7; 10-Q.pdf, p.31; 10-K.pdf, p.11).  
- Supporting products: digital engagement, analytics (DataHub, InfoCenter), AI capabilities, and the Guidewire Marketplace with partner apps and integrations to extend functionality (10-K.pdf, p.15; 10-K.pdf, p.7).  
- Delivery & commercial model: pro